# 🚀 Fine-Tune Qwen 2.5 on Free Google Colab (T4 GPU) & Export to Ollama (GGUF)

This notebook fine-tunes **Qwen 2.5 (3B or 7B)** on your document extraction dataset in ~10 minutes using **Unsloth (4-bit QLoRA)**, then exports it directly into a **GGUF file** optimized for fast CPU inference in **Ollama**.

## 1. Install Unsloth & Dependencies

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2. Load Base Model (Qwen 2.5 7B or 3B 4-bit)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 4096
# Choose 'Qwen2.5-3B-Instruct-bnb-4bit' (super fast on CPU) or 'Qwen2.5-7B-Instruct-bnb-4bit' (higher capability)
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 3. Upload `train.jsonl` Dataset
Run the cell below and select your `train.jsonl` file generated by the project.

In [ ]:
from google.colab import files
import os

if not os.path.exists("train.jsonl"):
    print("Please upload your train.jsonl file:")
    uploaded = files.upload()
else:
    print("train.jsonl already present.")

## 4. Format Dataset & Train

In [ ]:
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
        for convo in convos
    ]
    return {"text": texts}

dataset = load_dataset("json", data_files={"train": "train.jsonl"})
dataset = dataset.map(formatting_prompts_func, batched=True)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_ratio=0.1,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not FastLanguageModel.is_bfloat16_supported(),
        bf16=FastLanguageModel.is_bfloat16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        output_dir="output",
        report_to="none",
    ),
)

trainer_stats = trainer.train()

## 5. Export Directly to Ollama GGUF (4-bit Q4_K_M for CPU Inference)

In [ ]:
# Export to 4-bit GGUF format
model.save_pretrained_gguf("qwen2.5_doc_extractor_gguf", tokenizer, quantization_method="q4_k_m")

print("\n=== GGUF Export Complete! ===")

## 6. Download the GGUF Model to Your Local PC

In [ ]:
import glob
from google.colab import files

gguf_files = glob.glob("qwen2.5_doc_extractor_gguf/*.gguf")
if gguf_files:
    print(f"Downloading {gguf_files[0]}...")
    files.download(gguf_files[0])
else:
    print("No GGUF file found. Check Step 5.")